# EchoInsight V2 Demo

This notebook runs a very small EchoInsight V2 demo and inspects the resulting outputs.

What it does:

1. Configure the project root, Python executable, credential file, and model alias.
2. Optionally list the available model aliases.
3. Run a small pipeline demo on `data/airpod.csv`.
4. Inspect `v2_summary.json`, `feature_map.csv`, the initialized feature corpus, and review diagnostics.

Before running the pipeline cell, make sure `INFO_PATH` points to your local credential file.

In [ ]:
from pathlib import Path
import json
import csv
import subprocess
import sys

# Adjust these values if your local environment is different.
PROJECT_ROOT = Path.cwd()
PYTHON_EXE = sys.executable

# Update this to your local credential file.
INFO_PATH = PROJECT_ROOT / "api" / "infor_zhipu.md"

# Choose a model alias from config/model_registry.json.
MODEL_ALIAS = "glm-4.7-zhipu"

CSV_PATH = PROJECT_ROOT / "data" / "airpod.csv"
RUN_NAME = "notebook_demo_airpod"
RESULTS_DIR = PROJECT_ROOT / "results_v2" / RUN_NAME

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PYTHON_EXE:", PYTHON_EXE)
print("INFO_PATH:", INFO_PATH)
print("MODEL_ALIAS:", MODEL_ALIAS)
print("CSV_PATH:", CSV_PATH)
print("RESULTS_DIR:", RESULTS_DIR)

In [ ]:
# Optional: inspect the available model aliases before choosing MODEL_ALIAS.
subprocess.run(
    [PYTHON_EXE, "run_v2.py", "--list-models"],
    cwd=PROJECT_ROOT,
    check=True,
)

In [ ]:
# Small demo run. This is intentionally tiny so it finishes quickly.
cmd = [
    PYTHON_EXE,
    "run_v2.py",
    "--csv", str(CSV_PATH),
    "--info", str(INFO_PATH),
    "--model", MODEL_ALIAS,
    "--run-name", RUN_NAME,
    "--max-reviews", "3",
    "--sample-size", "3",
    "--max-features", "5",
    "--classify-workers", "1",
    "--max-validation-iters", "2",
]

print("Running command:\n")
print(" ".join(cmd))

subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)

## Inspect the summary

In [ ]:
summary_path = RESULTS_DIR / "v2_summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8"))
summary

## Preview the feature map

In [ ]:
feature_map_path = RESULTS_DIR / "feature_map.csv"
with feature_map_path.open(newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    rows = []
    for idx, row in enumerate(reader):
        rows.append(row)
        if idx >= 2:
            break
rows

## Inspect the initialized feature corpus

In [ ]:
init_corpus_path = RESULTS_DIR / "initialized_feature_corpus.json"
init_corpus = json.loads(init_corpus_path.read_text(encoding="utf-8"))
len(init_corpus), init_corpus[:5]

## Inspect review-level diagnostics

In [ ]:
diag_path = RESULTS_DIR / "review_level_diagnostics.jsonl"
diag_rows = []
with diag_path.open(encoding="utf-8") as f:
    for idx, line in enumerate(f):
        diag_rows.append(json.loads(line))
        if idx >= 1:
            break
diag_rows

## Next steps

- Increase `--max-reviews` from `3` to `10` or `20` once the smoke demo works.
- Try another model alias by changing `MODEL_ALIAS`.
- Open `report.md` and `feature_stats_report.md` under the run directory for more readable summaries.
- If the run already finished but summary files need regeneration, use `scripts/finalize_existing_run.py` or `scripts/summarize_results.py`.